# GX_06
# Higher order modes in waveguides

This graded exercise will expand on how we can use the beam propagation method to look at the propagation of higher order modes in a waveguide and how they react to a defect in the waveguide.

In [ ]:
# - No modification necessary -

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import fsolve
from scipy.fft import fft, ifft, fftfreq
from ipywidgets import interact, FloatSlider

# -----------------------
# Global simulation params
# -----------------------
wavelength = 800e-9
k0 = 2 * np.pi / wavelength

# Spatial grid
Nx = 2048
Nz = 500
x_max = 15e-6
z_max = 100e-6

x = np.linspace(-x_max, x_max, Nx)
z = np.linspace(0, z_max, Nz)

dx = x[1] - x[0]
dz = z[1] - z[0]

# Refractive indices
n_core = 1.6
n_clad = 1.5

In [ ]:
# - No modification necessary -

def create_slab_waveguide(width, n_core, n_clad, z_profile=None):
    n = np.ones((Nz, Nx)) * n_clad
    
    for i in range(Nz):
        w = width if z_profile is None else z_profile(z[i])
        mask = np.abs(x) <= w / 2
        n[i, mask] = n_core
    
    return n


def plot_waveguide(n):
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    
    # Cross-section
    ax[0].plot(x * 1e6, n[0])
    ax[0].set_title("Waveguide Cross-section")
    ax[0].set_xlabel("x (µm)")
    ax[0].set_ylabel("Refractive Index")
    
    # 2D view
    im = ax[1].imshow(n.T,
                  extent=[z[0]*1e6, z[-1]*1e6,
                          x[0]*1e6, x[-1]*1e6],
                  aspect='auto',
                  origin='lower',
                  cmap='viridis')

    ax[1].set_xlabel("z (µm)")
    ax[1].set_ylabel("x (µm)")
    ax[1].set_title("Waveguide (z-x)")
    
    plt.colorbar(im, ax=ax[1])
    plt.tight_layout()
    plt.show()

In [ ]:
# - No modification necessary -

width = 2e-6
n = create_slab_waveguide(width, n_core, n_clad)
plot_waveguide(n)

# Part 1

In this part you are going to work through how we can build an analytical solver for multiple modes of the waveguide -- not just the fundamental. 

To do this, you are going to begin by considering the V number of the waveguide, which is given as
$$
\frac{\text{NA} \pi d}{\lambda}
$$
for numerical aperture NA, core width d, and wavelength $\lambda$.

The V number allows us to solve for the modes that a waveguide will support as solutions of either
$$
u^2 + v^2  - V^2 = u * \tan{u} - v
$$
or
$$
u^2 + v^2  - V^2 = -\frac{u}{\tan{u}} - v
$$
for dimensionless parameters u and v related to real values by 
$$
u = \frac{w}{2} \sqrt{(n_{core} k_0)^2 - \beta^2}
$$
and
$$
v = \frac{w}{2} \sqrt{\beta^2 - (n_{clad} k_0)^2}
$$
Here u is proportional to the rate of oscillation in the core and v is proportional to the rate of decay in the cladding.

Finish the provided plotting code below, and use it to answer the following questions:
1) How many modes does this waveguide support?
2) As the number of the mode increases, what happens to the rate of oscillation in the core and rate of exponential decay in the cladding?

In [ ]:
NA = TODO
v_num = TODO

# Define grids in x and y dimension
v = np.linspace(0.0, 5, 200)
u = np.linspace(0.0, 5, 200)

# Span meshgrid over entire x/y plane
vv, uu = np.meshgrid(v, u)

# --- Mask problematic regions ---
eps = 1e-12
tan_uu = np.tan(uu)

# Mask where tan(u) is ~0 OR blows up (cos(u) ~ 0)
mask = (np.abs(tan_uu) < eps) | (np.abs(np.cos(uu)) < eps)

tan_uu = np.ma.masked_where(mask, tan_uu)

# Evaluate functions safely
V_p = vv**2 + uu**2 - v_num**2
gg = uu * tan_uu - vv
mm = -vv - uu / tan_uu

# Also mask outputs consistently
gg = np.ma.masked_where(mask, gg)
mm = np.ma.masked_where(mask, mm)

# Plot
fig, ax = plt.subplots(figsize=(8, 8))
ax.contour(uu, vv, V_p, [0], linewidths=1)
ax.contour(uu, vv, gg, linewidths=1)
ax.contour(uu, vv, mm, linewidths=1)

ax.set_title(r'Graphic Solution of transverse electric field', fontsize=16)
ax.set_xlabel('u', fontsize=12)
ax.set_ylabel('v', fontsize=12)
ax.set_xlim([0,5])
ax.set_ylim([0,5])

plt.show()

## Discussion
TODO

# Part 2

In this part, you will solve for the modes that are supported by the cavity and plot their relative profiles. After you have extracted the modes, you will try propagating them along the waveguide to verify that they are indeed modes.

To start, run the provided compute_slab_modes function. Be sure to solve for all modes you believe should be present. You should take some time to understand what is being done here, but will not be asked to implement any of this yourself. Then run the provided visualization code to check that your results align with your expectations from part 1.

After you have solved analytically for your modes, run the provided propagation code to try see how each of the modes propagates down the waveguide. Then, answer the following questions:

1) Have we successfully solved for modes of the waveguide?
2) What do the $\beta$ values provided with each mode mean? Why do they decrease with increasing mode number?

In [ ]:
# - No modification necessary -

def compute_slab_modes(x, n, width, n_core, n_clad, k0, v_num, num_modes=1):
    tol = 1e-3

    # -----------------------
    # Region splitting (same logic, cleaner)
    # -----------------------
    mask = n[0] > n_clad + tol
    idx = np.where(mask)[0]
    i1, i2 = idx[0], idx[-1] + 1

    x_clad_1 = x[:i1]
    x_core   = x[i1:i2]
    x_clad_2 = x[i2:]

    # -----------------------
    # EVEN MODES (same as original)
    # -----------------------
    Number = max(5, num_modes)  # ensure enough roots
    Se = np.zeros((Number, 1))

    fe = lambda u: np.abs(u)*np.tan(np.abs(u)) - np.sqrt(v_num**2 - u**2)

    initial_guess = 0
    for i in range(Number):
        initial_guess += 1
        Se[i, 0] = fsolve(fe, initial_guess)

    u_e = np.unique(Se.flatten())

    modes = []

    for u in u_e:
        v = np.sqrt(v_num**2 - u**2)
        h = 2 * u / width
        q = 2 * v / width

        beta = np.sqrt((n_core * k0)**2 - h**2)

        # build EVEN mode (same formula)
        B = 1
        D = np.cos(h * width / 2) / np.exp(-q * width / 2) * B
        C = D

        field = np.concatenate([
            D * np.exp(q * x_clad_1),
            B * np.cos(h * x_core),
            C * np.exp(-q * x_clad_2)
        ])

        modes.append((beta, field))

    # -----------------------
    # ODD MODES (fixed)
    # -----------------------
    if v_num > np.pi / 2:
        So = np.zeros((Number, 1))
        fo = lambda u: np.abs(u)/(np.tan(np.abs(u))) + np.sqrt(v_num**2 - u**2)

        for i in range(Number):
            So[i, 0] = fsolve(fo, 2.5)

        u_o = np.unique(So.flatten())

        for u in u_o:
            v = np.sqrt(v_num**2 - u**2)
            h = 2 * u / width
            q = 2 * v / width

            beta = np.sqrt((n_core * k0)**2 - h**2)

            # build ODD mode (correct scalar usage)
            B = 1
            D = np.sin(h * width / 2) / np.exp(-q * width / 2) * B
            C = D

            field = np.concatenate([
                -D * np.exp(q * x_clad_1),
                B * np.sin(h * x_core),
                C * np.exp(-q * x_clad_2)
            ])

            modes.append((beta, field))

    # -----------------------
    # Sort like physics (guided modes)
    # -----------------------
    modes.sort(key=lambda m: -m[0])  # highest beta first

    return modes[:num_modes]

In [ ]:
modes = compute_slab_modes(
    x=x,
    n=n,
    width=width,
    n_core=n_core,
    n_clad=n_clad,
    k0=k0,
    v_num=v_num,
    num_modes=TODO
)

# Unpack
for i, (beta, field) in enumerate(modes):
    print(f"Mode {i}: beta = {beta:.4e}, n_eff = {beta/k0:.2f}")
    plt.plot(x * 1e6, field, label=f"Mode {i}")

plt.legend()
plt.title("Slab Waveguide Modes")
plt.xlabel("x (µm)")
plt.show()

In [ ]:
# - No modification necessary -

def bpm_propagate(field0, n, n_ref):
    field = field0.copy()
    result = np.zeros((Nz, Nx), dtype=complex)
    
    kx = 2 * np.pi * fftfreq(Nx, d=dx)
    
    for i in range(Nz):
        result[i] = field
        
        # Diffraction (Fourier step)
        F = fft(field)
        F *= np.exp(-1j * (kx**2) * dz / (2 * k0 * n_ref))
        field = ifft(F)
        
        # Refraction
        phase = np.exp(1j * k0 * (n[i] - n_ref) * dz)
        field *= phase
    
    return result

In [ ]:
# - No modification necessary -

# Convert coordinates to micrometers
x_um = x * 1e6
z_um = z * 1e6

fig, axes = plt.subplots(1, len(modes), figsize=(15, 4), sharey=True)

if len(modes) == 1:
    axes = [axes]

for idx, (beta, field) in enumerate(modes):
    
    # Normalize input field
    field0 = field / np.sqrt(np.sum(np.abs(field)**2))
    
    # Propagate
    result = bpm_propagate(field0, n, n_ref=n_clad)
    
    intensity = np.abs(result)**2
    
    # Transpose so: horizontal = z, vertical = x
    intensity_plot = intensity.T
    
    ax = axes[idx]
    im = ax.imshow(
        intensity_plot,
        extent=[z_um[0], z_um[-1], x_um[0], x_um[-1]],
        aspect='auto',
        cmap='inferno',
        origin='lower'   # keeps x increasing upward
    )
    
    ax.set_title(f"Mode {idx+1}\nβ = {beta:.3e}")
    ax.set_xlabel("z (μm)")
    
    if idx == 0:
        ax.set_ylabel("x (μm)")
    
    # Individual colorbar
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("Intensity")

plt.tight_layout()
plt.show()

## Discussion
TODO

# Part 3
In this part, you will investigate how a defect in the waveguide affects the modes that you have already solved for.

To do this, you will be responsible for implementing the create_defective_slab_waveguide function, which will mirror the create_slab_waveguide function that you created in CW_06. Here, however, you should make it so that between the z coordinates defect_z_start and defect_z_end, there is a chip taken out of the top of the waveguide's core where the refractive index is set to that of the cladding. The depth of that chip is set by defect_depth_fraction, which is a fraction relative to the width of the complete core. Ultimately, your waveguide should look like it did before, but with a small rectangular piece of the core missing from the top edge of the waveguide.

After you have implemented the create_defective_slab_waveguide function, use the provided plotting code to visualize what you have created and make sure it matches with this description.

Then use the provided plotting code to propagate each of the eigenmodes of the original waveguide through your defective waveguide. For each mode, answer the following question: How can you explain the effect of the defect in terms of the affected field being a superposition of modes?

In [ ]:
def create_defective_slab_waveguide(width, n_core, n_clad,
                                    defect_z_start,
                                    defect_z_end,
                                    defect_depth_fraction=1/3):
    """
    Creates a slab waveguide with a rectangular defect on one side.

    defect_depth_fraction: fraction of core thickness removed (e.g. 1/3)
    defect_z_start, defect_z_end: where along z the defect exists
    """
    raise NotImplementedError

In [ ]:
# - No modification necessary -

# Define defect location (in meters)
defect_z_start = z[0] + 0.1 * (z[-1] - z[0])
defect_z_end   = z[0] + 0.2 * (z[-1] - z[0])

n_defect = create_defective_slab_waveguide(
    width,
    n_core,
    n_clad,
    defect_z_start,
    defect_z_end,
    defect_depth_fraction=1/3
)

plot_waveguide(n_defect)

In [ ]:
# - No modification necessary -

fig, axes = plt.subplots(1, len(modes), figsize=(15, 4), sharey=True)

if len(modes) == 1:
    axes = [axes]

for idx, (beta, field) in enumerate(modes):
    
    # Normalize
    field0 = field / np.sqrt(np.sum(np.abs(field)**2))
    
    # Propagate in defective structure
    result = bpm_propagate(field0, n_defect, n_ref=n_clad)
    
    intensity = np.abs(result)**2
    
    # Transpose for plotting (z horizontal)
    intensity_plot = intensity.T
    
    ax = axes[idx]
    im = ax.imshow(
        intensity_plot,
        extent=[z_um[0], z_um[-1], x_um[0], x_um[-1]],
        aspect='auto',
        cmap='inferno',
        origin='lower'
    )
    
    ax.set_title(f"Mode {idx+1}")
    ax.set_xlabel("z (μm)")
    
    if idx == 0:
        ax.set_ylabel("x (μm)")
    
    # Colorbar per subplot
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("Intensity")

plt.tight_layout()
plt.show()


## Discussion
TODO

# Bonus 

Now that you have seen how higher order modes react to a defect, let's try experimenting with understanding more deeply what the effect of the defect was. Using the modes that you have calculated, try making an input field that propagates as similarly as possible to the field of the fundamental mode after it has passed through the defect. Summarize your results below.

In [ ]:
# - No modification necessary -

import ipywidgets as widgets
from ipywidgets import interact

def plot_custom_mode(a1, a2, a3, p1, p2, p3):
    # Build fields
    field1 = modes[0][1] * a1 * np.exp(1j * p1)
    field2 = modes[1][1] * a2 * np.exp(1j * p2)
    field3 = modes[2][1] * a3 * np.exp(1j * p3)

    custom_field = field1 + field2 + field3

    # Normalize
    custom_field = custom_field / np.sqrt(np.sum(np.abs(custom_field)**2))

    # Propagate
    result = bpm_propagate(custom_field, n_defect, n_ref=n_clad)

    intensity = np.abs(result)**2
    intensity_plot = intensity.T

    # Plot
    plt.figure(figsize=(6,4))
    plt.imshow(
        intensity_plot,
        extent=[z_um[0], z_um[-1], x_um[0], x_um[-1]],
        aspect='auto',
        cmap='inferno',
        origin='lower'
    )
    plt.title("Custom Mode")
    plt.xlabel("z (μm)")
    plt.ylabel("x (μm)")
    plt.colorbar()
    plt.show()


interact(
    plot_custom_mode,
    a1=widgets.FloatSlider(value=1, min=0, max=3, step=0.1),
    a2=widgets.FloatSlider(value=1, min=0, max=3, step=0.1),
    a3=widgets.FloatSlider(value=1, min=0, max=3, step=0.1),
    p1=widgets.FloatSlider(value=0, min=0, max=2*np.pi, step=0.1),
    p2=widgets.FloatSlider(value=0, min=0, max=2*np.pi, step=0.1),
    p3=widgets.FloatSlider(value=0, min=0, max=2*np.pi, step=0.1),
);

## Discussion
TODO
